In [1]:
import duckdb

In [2]:
from config import DATA_DIR, RAW_DIR, PROCESSED_DIR

In [3]:
train_file = (RAW_DIR / "train.csv").as_posix()
metadata_file = (RAW_DIR / "building_metadata.csv").as_posix()
weather_file = (RAW_DIR / "weather_train.csv").as_posix()
output_file = (
    PROCESSED_DIR / "electricity_data.parquet"
).as_posix()

con = duckdb.connect()
con.execute("SET memory_limit = '4GB'")
con.execute("SET threads = 2")

query = """
COPY (
    SELECT
        t.building_id,
        b.site_id,
        b.primary_use,
        b.square_feet,

        t.timestamp,
        EXTRACT(HOUR FROM t.timestamp) AS hour,
        EXTRACT(DOW FROM t.timestamp) AS day_of_week,
        EXTRACT(MONTH FROM t.timestamp) AS month,

        w.air_temperature,
        w.dew_temperature,
        w.wind_speed,

        t.meter_reading

    FROM read_csv_auto(?) AS t

    JOIN read_csv_auto(?) AS b
        ON t.building_id = b.building_id

    LEFT JOIN read_csv_auto(?) AS w
        ON b.site_id = w.site_id
        AND t.timestamp = w.timestamp

    WHERE t.meter = 0
      AND b.site_id <> 0

) TO ? (FORMAT PARQUET, COMPRESSION ZSTD)
"""

con.execute(
    query,
    [train_file, metadata_file, weather_file, output_file]
)

print("Processed dataset saved to:", output_file)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processed dataset saved to: D:/Big Data Project/data/processed/electricity_data.parquet


In [4]:
result = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(air_temperature) AS matched_temperature,
        ROUND(
            100.0 * COUNT(air_temperature) / COUNT(*), 2
        ) AS temperature_coverage_pct,
        MIN(timestamp) AS first_timestamp,
        MAX(timestamp) AS last_timestamp
    FROM read_parquet(?)
""", [output_file]).df()

display(result)

,total_rows,matched_temperature,temperature_coverage_pct,first_timestamp,last_timestamp
0,11152501,11105485,99.58,2016-01-01,2016-12-31 23:00:00
